# File to run experiments with various hyperparameters

In [1]:
import pandas as pd
import numpy as np
import torch
import strats
import datetime
import os
import csv
import gc
import pickle


print(datetime.datetime.now())


Initializing package . . . 😘
2025-03-31 12:45:45.433898


# Prepare Data

#### Some functions

In [2]:
def compute_trimmed_stats(values: pd.Series, low_pct: float, high_pct: float):
    """
    values: 해당 그룹의 value Series
    low_pct, high_pct: 잘라낼 분위수 (예: 0.01, 0.99)
    
    반환: (trimmed_mean, trimmed_std)
    """
    lower_bound = values.quantile(low_pct)
    upper_bound = values.quantile(high_pct)
    trimmed = values[(values >= lower_bound) & (values <= upper_bound)]
    return trimmed.mean(), trimmed.std()

def calculate_all_stats(df: pd.DataFrame):
    # 결과를 담을 리스트
    results = []

    # itemid별로 그룹화
    grouped = df.groupby('itemid')

    for item_id, group in grouped:
        vals = group['value']

        # 1) 전체(아웃라이어 제거 없음) 평균/표준편차
        orig_mean = vals.mean()
        orig_std  = vals.std()

        # 2) 상하위 1% 제거
        mean_1pct, std_1pct = compute_trimmed_stats(vals, 0.01, 0.99)

        # 3) 상하위 3% 제거
        mean_3pct, std_3pct = compute_trimmed_stats(vals, 0.03, 0.97)

        # 4) 상하위 5% 제거
        mean_5pct, std_5pct = compute_trimmed_stats(vals, 0.05, 0.95)

        # 결과 한 줄로 정리
        results.append({
            'itemid': item_id,
            'orig_mean': orig_mean,
            'orig_std': orig_std,
            'mean_1pct': mean_1pct,
            'std_1pct': std_1pct,
            'mean_3pct': mean_3pct,
            'std_3pct': std_3pct,
            'mean_5pct': mean_5pct,
            'std_5pct': std_5pct,
            'lower_1' : vals.quantile(0.01),
            'upper_1' : vals.quantile(0.99),
        })

    # 리스트를 DataFrame으로
    df_stats = pd.DataFrame(results)
    return df_stats

def process_static(static):
    # Treat only in-hospital death
    static.loc[static['hospital_expire_flag'] == 0, 'deathtime'] = None
    print(len(static[static['outtime'] + 1440 >= static['deathtime']]))
    static = static.loc[:,['hadm_id', 'deathtime']]
    static = static.rename(columns={'hadm_id': 'hadm_id', 'deathtime' : 'death_offset'})
    unique_static_ids = static['hadm_id'].unique()
    return static, unique_static_ids

#### Run code

In [ ]:
mimic_static = pd.read_feather('mimic_data_static.feather')
mimic_outcome, mimic_ids = process_static(mimic_static)

data_vital = pd.read_feather('mimic_data_vital.feather').rename(columns={'valuenum' : 'value'})
data_lab = pd.read_feather('mimic_data_lab.feather').rename(columns={'valuenum' : 'value'})
data_treatment = pd.read_feather('mimic_data_treatment.feather').rename(columns={'valuenum' : 'value'})

print(f'{data_vital['hadm_id'].nunique()} in Vital')
print(f'{data_lab['hadm_id'].nunique()} in Lab')
print(f'{data_treatment['hadm_id'].nunique()} in Treatment')

data = pd.concat([
    data_vital, 
    data_lab, 
    data_treatment
    ])
print(f'{data['hadm_id'].nunique()} in Data')

# Factorize the 'item' column and get the mapping
encoded_total, actual_class_total = pd.factorize(data['itemid'])
data.loc[:,'itemid'] = encoded_total.astype(int)
data['itemid'] = pd.to_numeric(data['itemid'], errors='coerce')

data = data.dropna(subset=['value'])
data_stats = calculate_all_stats(data)

item_dict_total = dict(zip(actual_class_total, range(len(actual_class_total))))
emb_idx_total = len(item_dict_total)

with open('item_dict_total.pkl', 'rb') as f:
    item_dict_total = pickle.load(f)

# 재현성 유지를 위해 seed를 고정하고 섞습니다.
np.random.seed(9871)
np.random.shuffle(mimic_ids)

# 예: 80%를 train, 나머지를 valid로 사용
train_size = int(len(mimic_ids) * 0.8)
valid_size = int(len(mimic_ids) * 1)
train_hadm_ids = mimic_ids[:train_size]
valid_hadm_ids = mimic_ids[train_size:valid_size]
test_hadm_ids = mimic_ids[valid_size:]

train_final_ids = train_hadm_ids
valid_final_ids = valid_hadm_ids
test_final_ids = test_hadm_ids

# (D) 최종 DF
train_df = data[data['hadm_id'].isin(train_final_ids)].copy().reset_index(drop=True)
valid_df = data[data['hadm_id'].isin(valid_final_ids)].copy().reset_index(drop=True)
test_df = data[data['hadm_id'].isin(test_final_ids)].copy().reset_index(drop=True)

train_outcome_df = mimic_outcome[mimic_outcome['hadm_id'].isin(train_final_ids)].copy().reset_index(drop=True)
valid_outcome_df = mimic_outcome[mimic_outcome['hadm_id'].isin(valid_final_ids)].copy().reset_index(drop=True)
test_outcome_df = mimic_outcome[mimic_outcome['hadm_id'].isin(test_final_ids)].copy().reset_index(drop=True)
print(datetime.datetime.now())

with open('clip_bounds.pkl', 'rb') as f:
    clip_bounds = pickle.load(f)
make_loaders_24 = strats.MakeLoadersMor(train_df, valid_df, test_df,
                                        train_outcome_df, valid_outcome_df, test_outcome_df, obs_window=24*60,
                                        step=24*60, batch_size=32, mask_length=12*60, mask_segment_count=8, query_type='dynamic')

scaler_dict_24, clip_bounds_24, train_loader_24, valid_loader_24, test_loader_24, emb_idx = make_loaders_24.run_all(clip_bounds=clip_bounds)


# (2) scaler_dict 저장
with open('scaler_dict_mimic.pkl', 'wb') as f:
    pickle.dump(scaler_dict_24, f)


print(datetime.datetime.now())
gc.collect()

4049
54344 in Vital
54333 in Lab
52543 in Treatment
54344 in Data
2025-03-31 12:45:59.375569
[MakeLoaders] Beginning job... 😤
[Binary] Detecting binary itemids ...
  found 15 binary itemids => {32, 33, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31}
[Trimming] Using predefined clip bounds
  trimmed => train=33627409, valid=8343841, test=0
  [train] hadm_id row count => mean=773.49, std=1078.38, min=31, max=25817, median=417.0
  [valid] hadm_id row count => mean=767.67, std=1078.95, min=20, max=21298, median=415.0
  [test] is empty, skipping stats.
[Scale] Scaling type : StandardScaler
  Scaling done.
[MakeLoadersMor] Creating mortality datasets ...
[TimeSeriesDatasetMortality_Outcome] Total 42856 stays used
=== Stay-wise Statistics Table ===
                    Stay
Statistic               
count       42856.000000
mean         6536.317015
min          1440.000000
max        229918.000000
std          8628.095900
median       3668.500000
[TimeSeriesDataset] Created 174881 samples. 

0

#### Vital

In [ ]:
data = data_vital

# Factorize the 'item' column and get the mapping
encoded_vital, actual_class_vital = pd.factorize(data['itemid'])
data.loc[:,'itemid'] = encoded_vital.astype(int)
data['itemid'] = pd.to_numeric(data['itemid'], errors='coerce')

data = data.dropna(subset=['value'])

item_dict_vital = dict(zip(actual_class_vital, range(len(actual_class_vital))))
emb_idx_vital = len(item_dict_vital)

train_df = data[data['hadm_id'].isin(train_final_ids)].copy().reset_index(drop=True)
valid_df = data[data['hadm_id'].isin(valid_final_ids)].copy().reset_index(drop=True)
test_df = data[data['hadm_id'].isin(test_final_ids)].copy().reset_index(drop=True)
print(datetime.datetime.now())

# make_loaders_24 = strats.MakeLoadersMor(train_df, valid_df, test_df,
#                                         train_outcome_df, valid_outcome_df, test_outcome_df, obs_window=24*60,
#                                         step=360, batch_size=16, mask_length=360, mask_segment_count=4, query_type='dynamic', trim_percentile=0.01)

# scaler_vital_dict_24, train_vital_loader_24, valid_vital_loader_24, test_vital_loader_24, emb_vital_idx = make_loaders_24.run_all()

make_loaders_48 = strats.MakeLoadersMor(train_df, valid_df, test_df,
                                        train_outcome_df, valid_outcome_df, test_outcome_df, obs_window=48*60,
                                        step=720, batch_size=16, mask_length=720, mask_segment_count=4, query_type='dynamic', trim_percentile=0.01)

scaler_vital_dict_48, train_vital_loader_48, valid_vital_loader_48, test_vital_loader_48, emb_vital_idx = make_loaders_48.run_all()

print(datetime.datetime.now())

print('[Finished] Vital samples created')
gc.collect()

#### Lab

In [ ]:
data = data_lab

# Factorize the 'item' column and get the mapping
encoded_lab, actual_class_lab = pd.factorize(data['itemid'])
data.loc[:,'itemid'] = encoded_lab.astype(int)
data['itemid'] = pd.to_numeric(data['itemid'], errors='coerce')

data = data.dropna(subset=['value'])

item_dict_lab = dict(zip(actual_class_lab, range(len(actual_class_lab))))
emb_idx_lab = len(item_dict_lab)

train_df = data[data['hadm_id'].isin(train_final_ids)].copy().reset_index(drop=True)
valid_df = data[data['hadm_id'].isin(valid_final_ids)].copy().reset_index(drop=True)
test_df = data[data['hadm_id'].isin(test_final_ids)].copy().reset_index(drop=True)
print(datetime.datetime.now())


# make_loaders_24 = strats.MakeLoadersMor(train_df, valid_df, test_df,
#                                         train_outcome_df, valid_outcome_df, test_outcome_df, obs_window=24*60,
#                                         step=360, batch_size=16, mask_length=360, mask_segment_count=4, query_type='dynamic', trim_percentile=0.01)

# scaler_lab_dict_24, train_lab_loader_24, valid_lab_loader_24, test_lab_loader_24, emb_lab_idx = make_loaders_24.run_all()

make_loaders_48 = strats.MakeLoadersMor(train_df, valid_df, test_df,
                                        train_outcome_df, valid_outcome_df, test_outcome_df, obs_window=48*60,
                                        step=720, batch_size=16, mask_length=720, mask_segment_count=4, query_type='dynamic', trim_percentile=0.01)

scaler_lab_dict_48, train_lab_loader_48, valid_lab_loader_48, test_lab_loader_48, emb_lab_idx = make_loaders_48.run_all()
print(datetime.datetime.now())
print('[Finished] Lab samples created')
gc.collect()

#### Treatment

In [ ]:
data = data_treatment

# Factorize the 'item' column and get the mapping
encoded_treatment, actual_class_treatment = pd.factorize(data['itemid'])
data.loc[:,'itemid'] = encoded_treatment.astype(int)
data['itemid'] = pd.to_numeric(data['itemid'], errors='coerce')

data = data.dropna(subset=['value'])

item_dict_treatment = dict(zip(actual_class_treatment, range(len(actual_class_treatment))))
emb_idx_treatment = len(item_dict_treatment)

train_df = data[data['hadm_id'].isin(train_final_ids)].copy().reset_index(drop=True)
valid_df = data[data['hadm_id'].isin(valid_final_ids)].copy().reset_index(drop=True)
test_df = data[data['hadm_id'].isin(test_final_ids)].copy().reset_index(drop=True)
print(datetime.datetime.now())

# make_loaders_24 = strats.MakeLoadersMor(train_df, valid_df, test_df,
#                                         train_outcome_df, valid_outcome_df, test_outcome_df, obs_window=24*60,
#                                         step=360, batch_size=16, mask_length=360, mask_segment_count=4, query_type='dynamic', trim_percentile=0.01)

# scaler_treatment_dict_24, train_treatment_loader_24, valid_treatment_loader_24, test_treatment_loader_24, emb_treatment_idx = make_loaders_24.run_all()

make_loaders_48 = strats.MakeLoadersMor(train_df, valid_df, test_df,
                                        train_outcome_df, valid_outcome_df, test_outcome_df, obs_window=48*60,
                                        step=720, batch_size=16, mask_length=720, mask_segment_count=4, query_type='dynamic', trim_percentile=0.01)

scaler_ltreatment_dict_48, train_treatment_loader_48, valid_treatment_loader_48, test_treatment_loader_48, emb_treatment_idx = make_loaders_48.run_all()

print(datetime.datetime.now())
print('[Finished] Treatment samples created')
gc.collect()

# Experiments

### Define class

In [ ]:
class STraTSExperiment:
    """
    STraTS 모델 전체 파이프라인을 하나의 클래스로 묶어,
    - 모델 파라미터 설정
    - Pretrain
    - Pretrain Validate
    - Downstream Train
    - Evaluate
    등을 메서드로 구성.
    """
    def __init__(self,
                 emb_idx,
                 train_loader,
                 valid_loader,
                 test_loader,
                 version=None,
                 # 모델 관련 파라미터
                 num_heads=4,
                 num_layers=2,
                 ff_dim=64,
                 embed_dim=32,
                 dropout=0.2,
                 time_activation='relu',
                 value_activation='tanh',
                 final_emb_type='cls',
                 fusion_emb_weight=0.5,
                 final_emb_weight=0.5,
                 loss_type='bce',
                 # 학습 관련 파라미터
                 patience=30,
                 device=None):
        """
        emb_idx: 최종 임베딩 개수 (num_features)
        version: 모델/결과 파일 저장에 쓸 버전 이름 (기본값: 날짜_0 형태)
        기타 모델 하이퍼파라미터들...
        """
        # 버전 설정
        if version is None:
            # 날짜_0 형태 기본
            ver = 0
            today = datetime.date.today().isoformat()  # 예: '2024-01-05'
            version = f'{today}_{ver}'
        self.version = version
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.test_loader = test_loader

        # device
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.device = device

        # 모델 생성
        self.model = strats.STraTSModel(
            num_features=emb_idx,
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_blocks=num_layers,
            ff_dim=ff_dim,
            dropout=dropout,
            time_activation=time_activation,
            value_activation=value_activation,
            final_emb_type=final_emb_type,
            fusion_emb_weight=fusion_emb_weight,
            final_emb_weight=final_emb_weight,
            loss_type=loss_type
        ).to(self.device)

        # 학습 관련 파라미터
        self.patience = patience

        # optimizer (사용자가 바꿀 수 있도록)

        print(f"[STraTSExperiment] Initialized with version={self.version}, device={self.device}.")

    def pretrain(self, epochs=1000,
                 model_save_dir='./models_pt', learning_rate=1e-3):
        """
        Pretrain the model (masking-based).
        """
        # ensure save dir
        os.makedirs(model_save_dir, exist_ok=True)

        # pretrain
        print("[STraTSExperiment] Starting pretrain...")
        save_path = os.path.join(model_save_dir, f'pretrained_model_{self.version}.pt')

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=learning_rate)

        strats.pretrain_model(model=self.model,
                              train_loader=self.train_loader,
                              valid_loader=self.valid_loader,
                              epochs=epochs,
                              optimizer=optimizer,
                              device=self.device,
                              patience=self.patience,
                              model_save_path=save_path)

        print("[STraTSExperiment] Pretrain finished.")
        print(f"[STraTSExperiment] Best pretrain model saved at: {save_path}")
        self.model.load_state_dict(torch.load(save_path)) # Load best pretrain model

    def validate_pretrain(self, result_save_dir='./model_results'):
        """
        Validate pretrain results -> CSV
        """
        os.makedirs(result_save_dir, exist_ok=True)
        # 로드된 self.model 이용
        df_pretrain_results = strats.validate_model(self.model, self.valid_loader, device=self.device)
        # emb_idx에 해당하는 패딩/불필요 row 제거
        df_pretrain_results = df_pretrain_results.loc[df_pretrain_results['Variable'] != emb_idx]
        # CSV 저장
        save_path = os.path.join(result_save_dir, f'pretrained_result_{self.version}.csv')
        df_pretrain_results.to_csv(save_path, index=False)
        print(f"[STraTSExperiment] Pretrain validation result saved to {save_path}")

    def load_pretrained_model(self, model_load_path):
        """
        명시적으로 pretrained 모델을 로드하고 싶다면 호출
        """
        self.model.load_state_dict(torch.load(model_load_path, map_location=self.device))
        print(f"[STraTSExperiment] Loaded pretrained model from {model_load_path}")

    def train_downstream(self,
                         epochs=1000,
                         freeze=False,
                         model_save_dir='./models_pt',
                         result_save_dir='./model_results',
                         learning_rate = 1e-4):
        """
        Downstream train (fine-tuning or freeze)
        - freeze=True => pretrain 부분 고정
        - use_loss => ['death'], ['saps', 'sofa', 'death'] 등
        """
        os.makedirs(model_save_dir, exist_ok=True)
        os.makedirs(result_save_dir, exist_ok=True)
        save_path = os.path.join(model_save_dir, f'downstream_model_{self.version}.pt')

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=learning_rate)


        print("[STraTSExperiment] Starting downstream training...")
        df_downstream_loss = strats.train_model(
            model=self.model,
            train_loader=self.train_loader,
            valid_loader=self.valid_loader,
            epochs=epochs,
            optimizer=optimizer,
            device=self.device,
            patience=self.patience,
            freeze=freeze,
            model_save_path=save_path,
        )
        # CSV로 저장
        csv_path = os.path.join(result_save_dir, f'downstream_loss_{self.version}.csv')
        df_downstream_loss.to_csv(csv_path, index=False)
        print("[STraTSExperiment] Downstream training finished.")
        print(f"[STraTSExperiment] Best downstream model saved at: {save_path}")
        print(f"[STraTSExperiment] Training loss saved to {csv_path}")
        self.model.load_state_dict(torch.load(save_path))

    def load_downstream_model(self, model_load_path):
        """
        명시적으로 downstream 모델 로드
        """
        self.model.load_state_dict(torch.load(model_load_path, map_location=self.device))
        print(f"[STraTSExperiment] Loaded downstream model from {model_load_path}")

    def evaluate(self, result_save_dir='./model_results'):
        """
        Evaluate the final model on test set => metrics (AUROC, AUPRC, etc.)
        Also save the (hadm_id, query_time, embedding) data to CSV.
        """
        os.makedirs(result_save_dir, exist_ok=True)

        # 이 evaluate_model 함수는 수정된 버전으로,
        # (metrics, test_embeddings)를 리턴한다고 가정
        metrics, test_embeddings = strats.evaluate_model(self.model, self.test_loader, self.device)

        # 1) 성능 지표 출력
        print("[STraTSExperiment] Test Metrics:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")

        with open(f'{result_save_dir}/test_metrics_{self.version}.csv','w') as f:
            w = csv.writer(f)
            w.writerow(metrics.keys())
            w.writerow(metrics.values())

        # 2) 임베딩 저장
        # test_embeddings = {
        #   'hadm_id': np.array([...]),        # shape (N,)
        #   'query_time': np.array([...]),     # shape (N,)
        #   'embedding': np.array([...])       # shape (N, embed_dim)
        # }
        hadm_ids = test_embeddings['hadm_id']
        query_times = test_embeddings['query_time']
        emb_array = test_embeddings['embedding']  # shape=(N, embed_dim)
        num_samples, embed_dim = emb_array.shape

        # 임베딩을 pandas DataFrame으로 변환
        # 각 row: hadm_id, query_time, emb_0, emb_1, ... emb_(embed_dim-1)
        embed_df = pd.DataFrame({
            'hadm_id': hadm_ids,
            'query_time': query_times
        })
        for i in range(embed_dim):
            embed_df[f'emb_{i}'] = emb_array[:, i]

        # CSV로 저장
        emb_save_path = os.path.join(result_save_dir, f'test_embedding_{self.version}.csv')
        embed_df.to_csv(emb_save_path, index=False)


        # Train embeddings
        _, train_embedding = strats.evaluate_model(self.model, self.train_loader, self.device)

        hadm_ids = train_embedding['hadm_id']
        query_times = train_embedding['query_time']
        emb_array = train_embedding['embedding']  # shape=(N, embed_dim)
        num_samples, embed_dim = emb_array.shape

        embed_df = pd.DataFrame({
            'hadm_id': hadm_ids,
            'query_time': query_times
        })
        for i in range(embed_dim):
            embed_df[f'emb_{i}'] = emb_array[:, i]

        # CSV로 저장
        emb_save_path = os.path.join(result_save_dir, f'train_embedding_{self.version}.csv')
        embed_df.to_csv(emb_save_path, index=False)

        # Valid...
        _, valid_embedding = strats.evaluate_model(self.model, self.valid_loader, self.device)

        hadm_ids = valid_embedding['hadm_id']
        query_times = valid_embedding['query_time']
        emb_array = valid_embedding['embedding']  # shape=(N, embed_dim)
        num_samples, embed_dim = emb_array.shape

        embed_df = pd.DataFrame({
            'hadm_id': hadm_ids,
            'query_time': query_times
        })
        for i in range(embed_dim):
            embed_df[f'emb_{i}'] = emb_array[:, i]

        # CSV로 저장
        emb_save_path = os.path.join(result_save_dir, f'valid_embedding_{self.version}.csv')
        embed_df.to_csv(emb_save_path, index=False)

        print(f"[STraTSExperiment] Saved embeddings to {emb_save_path}")


        return metrics
    

### Task 0

In [ ]:
# Pretrained on all variables, using 1 hour step
exp_0 = STraTSExperiment(
    emb_idx=emb_idx,            # num_features
    train_loader=train_loader_48,
    valid_loader=valid_loader_48,
    test_loader=test_loader_48,
    version='mimic_48h',
    num_heads=4,
    num_layers=2,
    ff_dim=64,
    embed_dim=32,
    dropout=0.2,
    time_activation='relu',
    value_activation='tanh',
    final_emb_type='fusion',
    fusion_emb_weight=0.5,
    final_emb_weight=0.5,
    patience=7,
    loss_type='bce'
)
print(f'✅ Starting at {datetime.datetime.now()}')
# 1) Pretrain
exp_0.pretrain(epochs=40, learning_rate=1e-3)
exp_0.validate_pretrain()
print(f'✅ Job done at {datetime.datetime.now()}')

In [ ]:
# Pretrained on vital variables
exp_1 = STraTSExperiment(
    emb_idx=emb_idx,            # num_features
    train_loader=train_vital_loader_48,
    valid_loader=valid_vital_loader_48,
    test_loader=test_vital_loader_48,
    version='mimic_48h_vital',
    num_heads=4,
    num_layers=2,
    ff_dim=64,
    embed_dim=32,
    dropout=0.2,
    time_activation='relu',
    value_activation='tanh',
    final_emb_type='fusion',
    fusion_emb_weight=0.5,
    final_emb_weight=0.5,
    patience=7,
    loss_type='bce'
)
print(f'✅ Starting at {datetime.datetime.now()}')
# 1) Pretrain
exp_1.pretrain(epochs=40, learning_rate=1e-3)
exp_1.validate_pretrain()
print(f'✅ Job done at {datetime.datetime.now()}')

In [ ]:
# Pretrained on lab variables
exp_2 = STraTSExperiment(
    emb_idx=emb_idx,            # num_features
    train_loader=train_lab_loader_48,
    valid_loader=valid_lab_loader_48,
    test_loader=test_lab_loader_48,
    version='mimic_48h_lab',
    num_heads=4,
    num_layers=2,
    ff_dim=64,
    embed_dim=32,
    dropout=0.2,
    time_activation='relu',
    value_activation='tanh',
    final_emb_type='fusion',
    fusion_emb_weight=0.5,
    final_emb_weight=0.5,
    patience=7,
    loss_type='bce'
)
print(f'✅ Starting at {datetime.datetime.now()}')
# 1) Pretrain
exp_2.pretrain(epochs=40, learning_rate=1e-3)
exp_2.validate_pretrain()
print(f'✅ Job done at {datetime.datetime.now()}')

In [ ]:
# Pretrained on treatment variables
exp_3 = STraTSExperiment(
    emb_idx=emb_idx,            # num_features
    train_loader=train_treatment_loader_48,
    valid_loader=valid_treatment_loader_48,
    test_loader=test_treatment_loader_48,
    version='mimic_48h_treatment',
    num_heads=4,
    num_layers=2,
    ff_dim=64,
    embed_dim=32,
    dropout=0.2,
    time_activation='relu',
    value_activation='tanh',
    final_emb_type='fusion',
    fusion_emb_weight=0.5,
    final_emb_weight=0.5,
    patience=7,
    loss_type='bce'
)
print(f'✅ Starting at {datetime.datetime.now()}')
# 1) Pretrain
exp_3.pretrain(epochs=40, learning_rate=1e-3)
exp_3.validate_pretrain()
print(f'✅ Job done at {datetime.datetime.now()}')